# 08 — Montar a base final (`df_cnpj_espacial`)

**Objetivo:** juntar, em um único DataFrame, a localização de cada CNPJ e os dados do estabelecimento, da empresa, do Simples/MEI, da CNAE e do cadastro municipal; acrescentar as descrições dos códigos da Receita e as datas de abertura e encerramento.

**Entradas:** `df_espacial` (07), `df_estab`, `df_empresas` e `df_simples` (01), `df_cnae` (02), `df_cadastro` (03) e os arquivos de descrição da Receita em `01_dados_brutos/receita_federal/` (`*NATJUCSV`, `*MOTICSV`, `*PAISCSV` e `*CNAECSV`).

**Saídas:** `03_processamento/08_montar_base_final/df_cnpj_espacial`.

**Como rodar:** de cima para baixo. Antes da junção, confere que o CNPJ é o mesmo linha a linha em `df_espacial` e `df_estab`.

**Seções:** 1. DataFrame único com todos os CNPJs · 2. Descrições dos códigos · 3. Datas de abertura e encerramento · 4. Gravação e validação · 5. Resumo


In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, datas, descricoes, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("08_montar_base_final", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# ===== PARÂMETROS (config.yaml) =====
CNAE_NAO_INFORMADO = cfg.parametro("cnae.nao_informado")   # código da Receita para atividade não informada
PASTA_RECEITA = cfg.caminho("brutos.receita_federal")      # onde estão os arquivos de descrição de códigos da Receita
DESCRICOES_RECEITA = cfg.parametro("descricoes_receita")   # padrão do nome e dígitos de cada arquivo de descrição
LEGENDAS = cfg.parametro("legendas_receita")               # nomes dos códigos do layout (situação, porte, matriz/filial, Simples e MEI)

# ===== ENTRADAS =====
df_espacial = fontes.ler_processamento(cfg, "espacial_07")    # 07
df_estab = fontes.ler_processamento(cfg, "df_estab")          # 01
df_empresas = fontes.ler_processamento(cfg, "df_empresas")    # 01
df_simples = fontes.ler_processamento(cfg, "df_simples")      # 01
df_cnae = fontes.ler_processamento(cfg, "df_cnae")            # 02
df_cadastro = fontes.ler_processamento(cfg, "df_cadastro")    # 03

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. DataFrame único com todos os CNPJs
Junta, em um único DataFrame (`df_cnpj_espacial`), os dados de `df_espacial` (coordenadas, método, ajuste ao limite e CEP corrigido), de estabelecimentos (`df_estab`, pela numeração de linha), de empresas (`df_empresas`, pelo `cnpj_basico`), do Simples/MEI (`df_simples`, pelo `cnpj_basico`), da CNAE (`df_cnae`, pelo `cnae_principal`) e do cadastro municipal (`df_cadastro`, pelo CNPJ de 14 dígitos).
Da CNAE entram a denominação da subclasse e o código e a denominação da seção e da divisão. O código `8888888` (não informado) fica com seção e divisão `ND`; códigos fora da tabela do IBGE ganham a divisão pelos 2 primeiros dígitos, quando ela existe. A coluna `cnae_origem` diz de onde veio a classificação. Os CNAEs secundários só ganham nomes na seção 2.
Do cadastro municipal entram, com o prefixo `cad_`, a inscrição, `estabelecido`, `tipo`, `reg_tributario`, o CNAE e sua descrição, a quantidade de inscrições do CNPJ e `cad_cnae_diverge` (o CNAE do cadastro difere do da Receita?). Os dois CNAEs são mantidos; nada do que veio da Receita é reescrito. Se um CNPJ tem mais de uma inscrição, vale a primeira linha da planilha.
Entram TODOS os CNPJs, inclusive os sem cadastro. Se houver chave repetida em empresas, Simples ou CNAE, o bloco para com uma mensagem, sem alterar nada.
Esta seção não converte nem formata nada (as descrições dos códigos vêm na seção 2 e as datas na seção 3), e os DataFrames de origem não são alterados. Não exibe linhas dos DataFrames, só logs.

In [ ]:
log.info("SEÇÃO 1: criando o DataFrame único com todos os CNPJs...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_b29 = time.time()

# ===================================================================
# ETAPA 1: CONFERIR O df_espacial
# ===================================================================
log.info("Etapa 1: conferindo o df_espacial...")

# Verdadeiro para quem tem latitude E longitude (só para o resumo; ninguém é descartado).
tem_coord = df_espacial["lat"].notna() & df_espacial["lon"].notna()

# df_espacial e df_estab precisam ter a MESMA numeração de linhas, porque a junção é feita por ela.
if not df_espacial.index.equals(df_estab.index):
    raise ValueError("Bloco interrompido: df_espacial e df_estab não têm a mesma numeração de linhas. Nada foi alterado.")
# As duas tabelas vêm de arquivos de etapas diferentes (07 e 01): confere também se o CNPJ de cada linha é o mesmo.
cnpj_estab_linha = (df_estab["cnpj_basico"] + df_estab["cnpj_ordem"] + df_estab["cnpj_dv"]).astype(str)
# Quantas linhas têm CNPJ diferente nas duas tabelas (o esperado é zero).
linhas_diferentes = int((df_espacial["cnpj"].astype(str).to_numpy() != cnpj_estab_linha.to_numpy()).sum())
if linhas_diferentes:
    raise ValueError(f"Seção interrompida: {linhas_diferentes:,} linhas com CNPJ diferente entre df_espacial e df_estab. "
                     "Rode de novo o fluxo a partir do 05 (python executar_tudo.py --de 05). Nada foi alterado.")
log.info("    CNPJ igual linha a linha em df_espacial e df_estab: confere!")

log.info(f"Etapa 1 concluída! {len(df_espacial):,} CNPJs no total: "
      f"{int(tem_coord.sum()):,} com coordenada e {int((~tem_coord).sum()):,} sem coordenada (todos entram).")

# ===================================================================
# ETAPA 2: CONFERIR SE O CRUZAMENTO É SEGURO
# ===================================================================
log.info("\nEtapa 2: conferindo se há chave repetida nas tabelas que serão cruzadas...")

# duplicated() marca as linhas repetidas (além da primeira de cada chave); sum() conta quantas são.
# Uma repetição multiplicaria as linhas no cruzamento.
rep_empresas = int(df_empresas["cnpj_basico"].duplicated().sum())
rep_simples = int(df_simples["cnpj_basico"].duplicated().sum())
rep_cnae = int(df_cnae["cnae_principal"].duplicated().sum())

if rep_empresas > 0 or rep_simples > 0 or rep_cnae > 0:
    raise ValueError(
        f"Bloco interrompido: há chave repetida (df_empresas: {rep_empresas:,}; "
        f"df_simples: {rep_simples:,}; df_cnae: {rep_cnae:,}). "
        "O cruzamento multiplicaria linhas. Nada foi alterado."
    )

log.info("Etapa 2 concluída! Nenhuma repetição: o cruzamento é seguro.")

# ===================================================================
# ETAPA 3: JUNTAR COM OS DADOS DOS ESTABELECIMENTOS (df_estab)
# ===================================================================
log.info("\nEtapa 3: juntando com os dados dos estabelecimentos (df_estab)...")

# join junta pela numeração das linhas (que é a mesma nos dois DataFrames).
base = df_espacial.join(df_estab, how="left")

log.info(f"Etapa 3 concluída! {len(base):,} linhas, {base.shape[1]} colunas.")

# ===================================================================
# ETAPA 4: JUNTAR COM AS EMPRESAS E O SIMPLES
# ===================================================================
log.info("\nEtapa 4: juntando com as empresas e o Simples Nacional / MEI...")

# Verdadeiro para quem tem registro em cada tabela (só para o resumo no final).
tem_empresa = base["cnpj_basico"].isin(df_empresas["cnpj_basico"])
tem_simples = base["cnpj_basico"].isin(df_simples["cnpj_basico"])

# validate="m:1": o pandas confere de novo que cada chave aparece uma só vez na tabela trazida.
base = base.join(df_empresas.set_index("cnpj_basico"), on="cnpj_basico", how="left", validate="m:1")
base = base.join(df_simples.set_index("cnpj_basico"), on="cnpj_basico", how="left", validate="m:1")

log.info(f"Etapa 4 concluída! {len(base):,} linhas, {base.shape[1]} colunas.")

# ===================================================================
# ETAPA 5: JUNTAR COM A CNAE
# ===================================================================
log.info("\nEtapa 5: juntando com a tabela de CNAE (só o CNAE principal) e tratando os códigos fora da tabela...")

# Verdadeiro para quem tem o código na tabela de CNAE do IBGE (conferido ANTES do cruzamento).
tem_cnae = base["cnae_principal"].isin(df_cnae["cnae_principal"])

# Cruza com a tabela do IBGE, igual a antes: cada código aparece uma vez só na tabela (validate confere).
base = base.join(df_cnae.set_index("cnae_principal"), on="cnae_principal", how="left", validate="m:1")

# Código de CNAE como texto limpo (vazio vira texto vazio), usado nas regras abaixo.
codigo_cnae = base["cnae_principal"].fillna("").astype(str).str.strip()

# Grupo 1: o código 8888888 da Receita, que não é uma subclasse da CNAE (significa "não informado").
nao_informado = ~tem_cnae & (codigo_cnae == CNAE_NAO_INFORMADO)

# Tabela auxiliar: para cada divisão (2 dígitos), a seção a que ela pertence e os nomes de seção e de divisão.
mapa_divisao = (df_cnae.drop_duplicates("cnae_divisao")
                .set_index("cnae_divisao")[["cnae_secao", "cnae_secao_denominacao", "cnae_divisao_denominacao"]])

# Grupo 2: código de 7 dígitos fora da tabela cujos 2 primeiros dígitos são uma divisão que existe na tabela.
divisao_do_codigo = codigo_cnae.str[:2]
pela_divisao = (~tem_cnae & ~nao_informado
                & codigo_cnae.str.fullmatch(r"\d{7}")
                & divisao_do_codigo.isin(mapa_divisao.index))

# Grupo 3: o que sobrar (código fora da tabela e sem divisão conhecida): fica sem classificação.
sem_classificacao = ~tem_cnae & ~nao_informado & ~pela_divisao

# Coluna nova: de onde veio a classificação do CNAE de cada CNPJ (a primeira condição verdadeira vale).
base["cnae_origem"] = np.select(
    [tem_cnae, nao_informado, pela_divisao],
    ["tabela IBGE", "não informado", "divisão pelos 2 primeiros dígitos"],
    default="fora da tabela do IBGE, sem classificação",
)

# --- Grupo 1: "Não informado" (seção e divisão marcadas como ND) ---
base.loc[nao_informado, "cnae_subclasse"] = "Não informado"
base.loc[nao_informado, "cnae_denominacao"] = "Não informado (o código 8888888 da Receita não é uma subclasse da CNAE)"
base.loc[nao_informado, ["cnae_secao", "cnae_divisao"]] = "ND"
base.loc[nao_informado, ["cnae_secao_denominacao", "cnae_divisao_denominacao"]] = "Não informado"

# --- Grupo 2: divisão e seção deduzidas pelos 2 primeiros dígitos ---
if pela_divisao.any():
    # Divisão de cada CNPJ desse grupo e os dados dela na tabela auxiliar.
    divisoes_grupo = divisao_do_codigo[pela_divisao]
    dados_divisao = mapa_divisao.loc[divisoes_grupo]
    # Código formatado como na CNAE (ex.: 5812300 vira 5812-3/00).
    cod = codigo_cnae[pela_divisao]
    base.loc[pela_divisao, "cnae_subclasse"] = (cod.str[:4] + "-" + cod.str[4] + "/" + cod.str[5:]).to_numpy()
    base.loc[pela_divisao, "cnae_denominacao"] = "Subclasse fora da tabela CNAE 2.3 do IBGE"
    base.loc[pela_divisao, "cnae_divisao"] = divisoes_grupo.to_numpy()
    base.loc[pela_divisao, "cnae_secao"] = dados_divisao["cnae_secao"].to_numpy()
    base.loc[pela_divisao, "cnae_secao_denominacao"] = dados_divisao["cnae_secao_denominacao"].to_numpy()
    base.loc[pela_divisao, "cnae_divisao_denominacao"] = dados_divisao["cnae_divisao_denominacao"].to_numpy()

log.info(f"Etapa 5 concluída! {len(base):,} linhas, {base.shape[1]} colunas.")
log.info("    Origem da classificação do CNAE:")
for nome_origem, qtd in base["cnae_origem"].value_counts().items():
    log.info(f"        {nome_origem}: {qtd:,}")
# Códigos fora da tabela que foram tratados (poucos): código e quantidade de CNPJs.
for rotulo, mascara in [("divisão pelos 2 primeiros dígitos", pela_divisao), ("sem classificação", sem_classificacao)]:
    if mascara.any():
        log.info(f"    Códigos tratados como '{rotulo}':")
        for cod_item, qtd in codigo_cnae[mascara].value_counts().items():
            log.info(f"        {cod_item if cod_item else '(vazio)'}: {qtd:,} CNPJs")

# ===================================================================
# ETAPA 6: JUNTAR COM O CADASTRO MUNICIPAL
# ===================================================================
log.info("\nEtapa 6: juntando com o cadastro municipal (só CNPJs; CPF e nomes não entram)...")

# O cadastro vem do notebook 03 (seção 4). Sem ele, o bloco para aqui, sem alterar nada.
if "df_cadastro" not in globals():
    raise NameError("Bloco interrompido: o df_cadastro não existe. Rode o notebook 03 (seção 4) antes. Nada foi alterado.")

# Cópia do cadastro, com a quantidade de inscrições municipais de cada CNPJ.
cad = df_cadastro.copy()
cad["n_inscricoes"] = cad.groupby("cnpj")["cnpj"].transform("size")
# Quando um CNPJ tem mais de uma inscrição, vale a PRIMEIRA linha da planilha (n_inscricoes mostra que há outras).
cad = cad.drop_duplicates("cnpj", keep="first")
# Colunas que vão para a base final (com o prefixo cad_), sem reescrever nada que veio da Receita.
colunas_cad = {"insc_mun": "cad_insc_mun", "estabelecido": "cad_estabelecido", "tipo": "cad_tipo",
               "reg_tributario": "cad_reg_tributario", "cnae_principal": "cad_cnae_principal",
               "descricao_cnae": "cad_descricao_cnae", "n_inscricoes": "cad_n_inscricoes"}
cad = cad.set_index("cnpj")[list(colunas_cad)].rename(columns=colunas_cad)

# Verdadeiro para quem tem cadastro municipal (só para o resumo).
tem_cadastro = base["cnpj"].isin(cad.index)
# validate="m:1": o pandas confere de novo que cada CNPJ aparece uma só vez no cadastro.
base = base.join(cad, on="cnpj", how="left", validate="m:1")

# Coluna nova: o CNAE do cadastro é diferente do da Receita? (só onde os dois existem; os dois CNAEs são mantidos)
ambos_cnae = base["cad_cnae_principal"].notna() & base["cnae_principal"].notna()
diverge = pd.Series(None, index=base.index, dtype=object)
diverge[ambos_cnae] = np.where(base.loc[ambos_cnae, "cad_cnae_principal"] != base.loc[ambos_cnae, "cnae_principal"], "sim", "não")
base["cad_cnae_diverge"] = diverge

log.info(f"Etapa 6 concluída! {len(base):,} linhas, {base.shape[1]} colunas.")
log.info(f"    CNPJs com cadastro municipal: {int(tem_cadastro.sum()):,} | sem cadastro: {int((~tem_cadastro).sum()):,}")
log.info(f"    CNPJs com mais de uma inscrição no cadastro (vale a primeira): {int((base['cad_n_inscricoes'] > 1).sum()):,}")
log.info(f"    CNAE do cadastro diferente do da Receita: {int((diverge == 'sim').sum()):,} de {int(ambos_cnae.sum()):,} comparáveis")

# ===================================================================
# ETAPA 7: FINALIZAR E CONFERIR
# ===================================================================
log.info("\nEtapa 7: finalizando e conferindo o resultado...")

# Guarda o resultado com o nome final.
df_cnpj_espacial = base

# Quantas colunas deveriam existir (sem repetir as chaves das tabelas trazidas).
colunas_esperadas = (df_espacial.shape[1] + df_estab.shape[1]
                     + (df_empresas.shape[1] - 1) + (df_simples.shape[1] - 1) + (df_cnae.shape[1] - 1)
                     # + cnae_origem + as colunas do cadastro (cad_...) + cad_cnae_diverge
                     + 1 + len(colunas_cad) + 1)

# Conferência da ordem: a linha i do resultado deve ser o estabelecimento i do df_estab.
mesma_ordem = (df_cnpj_espacial.index.equals(df_estab.index)
               and bool((df_cnpj_espacial["cnpj_basico"] == df_estab["cnpj_basico"]).all()))
# Conferência do CNPJ de 14 dígitos: tem que ser igual a básico + ordem + dígito verificador.
cnpj_confere = bool((df_cnpj_espacial["cnpj"] == df_cnpj_espacial["cnpj_basico"]
                     + df_cnpj_espacial["cnpj_ordem"] + df_cnpj_espacial["cnpj_dv"]).all())

log.info("Etapa 7 concluída!")

log.info("\n--- RESUMO DO DATAFRAME ÚNICO (df_cnpj_espacial) ---")
log.info(f"Resultado: {len(df_cnpj_espacial):,} linhas | {df_cnpj_espacial.shape[1]} colunas")
for nome, mascara in [("Com coordenada", tem_coord), ("Sem coordenada", ~tem_coord)]:
    n = int(mascara.sum())
    log.info(f"  {nome}: {n:,}")
    log.info(f"    com dados de empresa: {int((mascara & tem_empresa).sum()):,}")
    log.info(f"    com registro no Simples/MEI: {int((mascara & tem_simples).sum()):,}")
    log.info(f"    com CNAE na tabela do IBGE: {int((mascara & tem_cnae).sum()):,}")

log.info("\n--- CNPJs POR SEÇÃO DA CNAE (10 maiores) ---")
for nome_secao, qtd in df_cnpj_espacial["cnae_secao_denominacao"].value_counts().head(10).items():
    log.info(f"  {nome_secao}: {qtd:,}")

log.info("\n--- CONFERÊNCIAS ---")
log.info(f"Linhas do resultado = CNPJs do df_espacial: "
      f"{'confere' if len(df_cnpj_espacial) == len(df_espacial) else 'NÃO confere'}")
log.info(f"Mesma ordem de linhas do df_estab: {'confere' if mesma_ordem else 'NÃO confere'}")
log.info(f"CNPJ de 14 dígitos = básico + ordem + dígito verificador: {'confere' if cnpj_confere else 'NÃO confere'}")
log.info(f"CNPJs de 14 dígitos todos diferentes: {'confere' if df_cnpj_espacial['cnpj'].is_unique else 'NÃO confere'}")
log.info(f"Número de colunas ({df_cnpj_espacial.shape[1]}) = esperado ({colunas_esperadas}): "
      f"{'confere' if df_cnpj_espacial.shape[1] == colunas_esperadas else 'NÃO confere'}")
log.info(f"Nomes de colunas sem repetição: {'confere' if df_cnpj_espacial.columns.is_unique else 'NÃO confere'}")

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b29)}.")

## 2. Descrições dos códigos
Acrescenta, **ao lado de cada código**, uma coluna `<campo>_nome` com a descrição dele. Os códigos continuam como vieram da Receita:
- pelos arquivos de descrição da Receita (`01_dados_brutos/receita_federal/`; código na 1ª coluna e descrição na 2ª): `natureza_juridica_nome` (`*NATJUCSV`), `motivo_situacao_nome` (`*MOTICSV`), `pais_nome` (`*PAISCSV`), `cnae_principal_nome` e `cnae_secundaria_nome` (`*CNAECSV`, mais preciso que a tabela do IBGE; esta continua só para a seção e a divisão da CNAE);
- pelo layout da Receita (`parametros.legendas_receita` no config.yaml): `situacao_nome`, `porte_nome` e `matriz_filial_nome`, com a mesma escrita do Monitor de Negócios (Ativa, Baixada, Microempresa, Matriz...);
- `opcao_simples_nome` e `opcao_mei_nome`: Sim, Não ou Outros (registro do Simples com a opção em branco). CNPJ sem registro no arquivo do Simples fica vazio.
A CNAE secundária vira uma lista `código – descrição`, separada por ` | `. Código sem descrição na tabela recebe `(código fora da tabela)` (ou `(código fora do layout)`, nos nomes do layout) e é contado no log; código vazio fica sem nome. Só logs.


In [ ]:
log.info("SEÇÃO 2: descrições dos códigos da Receita...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_desc = time.time()

# Trabalha direto no df_cnpj_espacial (criado na seção 1); só acrescenta colunas.
if "df_cnpj_espacial" not in globals():
    raise NameError("Seção interrompida: o df_cnpj_espacial não existe. Rode a seção 1 antes. Nada foi alterado.")
if "tem_simples" not in globals():
    raise NameError("Seção interrompida: falta a marcação de quem tem registro no Simples (etapa 4 da seção 1). Rode a seção 1 antes.")

# ===================================================================
# ETAPA 1: LER OS ARQUIVOS DE DESCRIÇÃO DA RECEITA
# ===================================================================
log.info("Etapa 1: lendo os arquivos de descrição (código na 1ª coluna, descrição na 2ª)...")

# Um dicionário: nome da tabela -> descrições (código vira o "índice" e a descrição vira o valor).
tabelas_codigos = {}
for nome_tabela in ["natureza_juridica", "motivo_situacao", "pais", "cnae"]:
    regra = DESCRICOES_RECEITA[nome_tabela]          # padrão do nome do arquivo e número de dígitos do código
    arquivo = descricoes.achar_arquivo(PASTA_RECEITA, regra["padrao"])
    tabela, codificacao = descricoes.ler_tabela_codigos(arquivo, regra["digitos"], DESCRICOES_RECEITA["separador"],
                                                        DESCRICOES_RECEITA["codificacoes"])
    tabelas_codigos[nome_tabela] = tabela
    log.info(f"    {nome_tabela}: {arquivo.name} | {len(tabela):,} códigos | codificação {codificacao}")
    # Letras como "Ã" e "�" no meio de um texto indicam que a codificação lida não é a do arquivo.
    if tabela.str.contains("Ã|�", regex=True).any():
        log.warning(f"    ATENÇÃO: {nome_tabela} tem letras estranhas (Ã ou �): confira os acentos e a codificação no config.yaml.")
log.info("Etapa 1 concluída!")

# ===================================================================
# ETAPA 2: NOMES DOS CÓDIGOS DO LAYOUT (SITUAÇÃO, PORTE, MATRIZ/FILIAL, SIMPLES E MEI)
# ===================================================================
log.info("\nEtapa 2: nomeando os códigos do layout da Receita...")

texto_fora_layout = LEGENDAS["texto_fora_do_layout"]       # "(código fora do layout)"
df_cnpj_espacial["situacao_nome"] = descricoes.nomear_por_dicionario(
    df_cnpj_espacial["situacao_cadastral"], LEGENDAS["situacao_cadastral"], texto_fora_layout)
df_cnpj_espacial["porte_nome"] = descricoes.nomear_por_dicionario(
    df_cnpj_espacial["porte_empresa"], LEGENDAS["porte_empresa"], texto_fora_layout)
df_cnpj_espacial["matriz_filial_nome"] = descricoes.nomear_por_dicionario(
    df_cnpj_espacial["matriz_filial"], LEGENDAS["matriz_filial"], texto_fora_layout)
# Simples e MEI: S = Sim, N = Não, em branco = Outros (só para quem tem registro no arquivo do Simples).
for coluna_opcao in ["opcao_simples", "opcao_mei"]:
    df_cnpj_espacial[coluna_opcao + "_nome"] = descricoes.nomear_sim_nao(
        df_cnpj_espacial[coluna_opcao], tem_simples, LEGENDAS["opcao_simples_mei"], LEGENDAS["texto_outros"], texto_fora_layout)

for coluna_nome in ["situacao_nome", "porte_nome", "matriz_filial_nome", "opcao_simples_nome", "opcao_mei_nome"]:
    log.info(f"    {coluna_nome}:")
    contagem = df_cnpj_espacial[coluna_nome].value_counts(dropna=False)
    for nome_item, qtd in contagem.items():
        log.info(f"        {nome_item if pd.notna(nome_item) else '(vazio)'}: {qtd:,}")
log.info("Etapa 2 concluída!")

# ===================================================================
# ETAPA 3: NOMES PELOS ARQUIVOS DE DESCRIÇÃO
# ===================================================================
log.info("\nEtapa 3: nomeando natureza jurídica, motivo, país e CNAE pelos arquivos da Receita...")


def nomear_coluna(coluna_codigo, coluna_nome, nome_tabela):
    """Cria a coluna de nomes de um código simples e conta no log os códigos que não têm descrição."""
    digitos = DESCRICOES_RECEITA[nome_tabela]["digitos"]
    tabela = tabelas_codigos[nome_tabela]
    # Código "limpo": só dígitos, com zeros à esquerda até o tamanho certo (o original não é alterado).
    codigos = descricoes.normalizar_codigo(df_cnpj_espacial[coluna_codigo], digitos)
    df_cnpj_espacial[coluna_nome] = descricoes.nomear(codigos, tabela)
    sem_descricao = descricoes.codigos_sem_descricao(codigos, tabela)
    log.info(f"    {coluna_nome}: {int(codigos.notna().sum() - sem_descricao.sum()):,} com descrição | "
             f"{int(codigos.isna().sum()):,} sem código (vazio) | "
             f"{int(sem_descricao.sum()):,} com código fora da tabela")
    for codigo_item, qtd in sem_descricao.head(10).items():
        log.info(f"        código sem descrição: {codigo_item}: {qtd:,} CNPJs")


nomear_coluna("natureza_juridica", "natureza_juridica_nome", "natureza_juridica")
nomear_coluna("motivo_situacao", "motivo_situacao_nome", "motivo_situacao")
nomear_coluna("pais", "pais_nome", "pais")
nomear_coluna("cnae_principal", "cnae_principal_nome", "cnae")

# CNAE secundária: uma lista "código – descrição | código – descrição" por CNPJ.
lista_secundaria, secundarias_fora = descricoes.lista_cnae_secundaria(df_cnpj_espacial["cnae_secundaria"], tabelas_codigos["cnae"])
df_cnpj_espacial["cnae_secundaria_nome"] = lista_secundaria
log.info(f"    cnae_secundaria_nome: {int(lista_secundaria.notna().sum()):,} CNPJs com CNAE secundária | "
         f"{int(secundarias_fora.sum()):,} códigos fora da tabela ({len(secundarias_fora):,} diferentes)")
for codigo_item, qtd in secundarias_fora.head(10).items():
    log.info(f"        código sem descrição: {codigo_item}: {qtd:,} ocorrências")
log.info("Etapa 3 concluída!")

# ===================================================================
# ETAPA 4: COLOCAR CADA NOME AO LADO DO SEU CÓDIGO
# ===================================================================
log.info("\nEtapa 4: colocando cada nome ao lado do seu código...")
df_cnpj_espacial = descricoes.colocar_ao_lado(df_cnpj_espacial, [
    ("matriz_filial", ["matriz_filial_nome"]),
    ("situacao_cadastral", ["situacao_nome"]),
    ("motivo_situacao", ["motivo_situacao_nome"]),
    ("pais", ["pais_nome"]),
    ("cnae_principal", ["cnae_principal_nome"]),
    ("cnae_secundaria", ["cnae_secundaria_nome"]),
    ("natureza_juridica", ["natureza_juridica_nome"]),
    ("porte_empresa", ["porte_nome"]),
    ("opcao_simples", ["opcao_simples_nome"]),
    ("opcao_mei", ["opcao_mei_nome"]),
])
log.info(f"Etapa 4 concluída! {len(df_cnpj_espacial):,} linhas, {df_cnpj_espacial.shape[1]} colunas.")

log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_desc)}.")


## 3. Datas de abertura e encerramento
Converte as datas da Receita (texto `AAAAMMDD`) em **datas de verdade** (`data_situacao`, `data_inicio_atividade`, `data_situacao_especial`, `data_opcao_simples`, `data_exclusao_simples`, `data_opcao_mei` e `data_exclusao_mei`). Zeros (`0` e `00000000`) e datas impossíveis viram vazio, sem inventar data.
Cria as colunas para a análise de aberturas e encerramentos:
- `data_abertura` (a `data_inicio_atividade`), `ano_abertura` e `ano_mes_abertura` (texto `AAAA-MM`);
- `data_encerramento`, `ano_encerramento` e `ano_mes_encerramento`: **só para estabelecimentos Baixados** (situação 8), com a `data_situacao`, que para eles é a data da baixa. Para os demais ficam vazias.
O log conta as datas vazias e as incoerências (encerramento antes da abertura). Só logs.


In [ ]:
log.info("SEÇÃO 3: datas de abertura e encerramento...")
# Guarda a hora de início para calcular o tempo total no final.
inicio_datas = time.time()

if "df_cnpj_espacial" not in globals():
    raise NameError("Seção interrompida: o df_cnpj_espacial não existe. Rode a seção 1 antes. Nada foi alterado.")

# ===================================================================
# ETAPA 1: CONVERTER AS DATAS DE TEXTO (AAAAMMDD) PARA DATA DE VERDADE
# ===================================================================
log.info("Etapa 1: convertendo as datas da Receita...")
COLUNAS_DATA = ["data_situacao", "data_inicio_atividade", "data_situacao_especial", "data_opcao_simples",
                "data_exclusao_simples", "data_opcao_mei", "data_exclusao_mei"]
for coluna_data in COLUNAS_DATA:
    if coluna_data not in df_cnpj_espacial.columns:
        continue
    df_cnpj_espacial[coluna_data] = datas.para_data(df_cnpj_espacial[coluna_data])
    com_data = int(df_cnpj_espacial[coluna_data].notna().sum())
    log.info(f"    {coluna_data}: {com_data:,} com data | {len(df_cnpj_espacial) - com_data:,} sem data (vazia ou 00000000)")
log.info("Etapa 1 concluída!")

# ===================================================================
# ETAPA 2: ABERTURA
# ===================================================================
log.info("\nEtapa 2: criando as colunas de abertura...")
df_cnpj_espacial["data_abertura"] = df_cnpj_espacial["data_inicio_atividade"]
df_cnpj_espacial["ano_abertura"] = datas.ano_da_data(df_cnpj_espacial["data_abertura"])
df_cnpj_espacial["ano_mes_abertura"] = datas.ano_mes_da_data(df_cnpj_espacial["data_abertura"])
log.info(f"Etapa 2 concluída! Abertura de {df_cnpj_espacial['data_abertura'].min():%d/%m/%Y} "
         f"a {df_cnpj_espacial['data_abertura'].max():%d/%m/%Y}.")

# ===================================================================
# ETAPA 3: ENCERRAMENTO (SÓ QUEM ESTÁ BAIXADO)
# ===================================================================
log.info("\nEtapa 3: criando as colunas de encerramento (só estabelecimentos Baixados)...")
# Verdadeiro para quem está Baixado (situação 8 no layout da Receita).
baixada = pd.to_numeric(df_cnpj_espacial["situacao_cadastral"], errors="coerce") == LEGENDAS["situacao_baixada"]
# Para quem está Baixado, a data da situação é a data da baixa; para os demais, fica vazio.
df_cnpj_espacial["data_encerramento"] = df_cnpj_espacial["data_situacao"].where(baixada)
df_cnpj_espacial["ano_encerramento"] = datas.ano_da_data(df_cnpj_espacial["data_encerramento"])
df_cnpj_espacial["ano_mes_encerramento"] = datas.ano_mes_da_data(df_cnpj_espacial["data_encerramento"])
log.info(f"Etapa 3 concluída! Baixados: {int(baixada.sum()):,}.")

# ===================================================================
# ETAPA 4: CONFERIR AS DATAS
# ===================================================================
log.info("\nEtapa 4: conferindo as datas...")
sem_abertura = int(df_cnpj_espacial["data_abertura"].isna().sum())
baixadas_sem_data = int((baixada & df_cnpj_espacial["data_encerramento"].isna()).sum())
antes_da_abertura = int((df_cnpj_espacial["data_encerramento"] < df_cnpj_espacial["data_abertura"]).sum())
log.info(f"    Sem data de abertura: {sem_abertura:,}")
log.info(f"    Baixados sem data da baixa: {baixadas_sem_data:,}")
log.info(f"    Encerramento ANTES da abertura (data com erro na Receita): {antes_da_abertura:,}")
(log.info if int(df_cnpj_espacial["data_encerramento"].notna().sum()) == int(baixada.sum()) - baixadas_sem_data else log.warning)(
    f"    Encerramentos preenchidos = baixados com data: {int(df_cnpj_espacial['data_encerramento'].notna().sum()):,}")
datas_todas = pd.concat([df_cnpj_espacial["data_situacao"], df_cnpj_espacial["data_inicio_atividade"]]).dropna()
log.info(f"    Data mais recente dos registros (data da base): {datas_todas.max():%d/%m/%Y}")
log.info("    Encerramentos por ano (últimos 5 anos):")
por_ano_enc = df_cnpj_espacial["ano_encerramento"].value_counts().sort_index()
for ano_item, qtd in por_ano_enc.tail(5).items():
    log.info(f"        {int(ano_item)}: {qtd:,}")
log.info("    Aberturas por ano (últimos 5 anos):")
por_ano_ab = df_cnpj_espacial["ano_abertura"].value_counts().sort_index()
for ano_item, qtd in por_ano_ab.tail(5).items():
    log.info(f"        {int(ano_item)}: {qtd:,}")
log.info("Etapa 4 concluída!")

# ===================================================================
# ETAPA 5: COLOCAR AS COLUNAS NOVAS AO LADO DAS DATAS DA RECEITA
# ===================================================================
log.info("\nEtapa 5: colocando as colunas novas ao lado das datas da Receita...")
df_cnpj_espacial = descricoes.colocar_ao_lado(df_cnpj_espacial, [
    ("data_situacao", ["data_encerramento", "ano_encerramento", "ano_mes_encerramento"]),
    ("data_inicio_atividade", ["data_abertura", "ano_abertura", "ano_mes_abertura"]),
])
# Conferência do número de colunas: as da seção 1 + 10 nomes (seção 2) + 6 colunas de abertura e encerramento.
esperado_final = colunas_esperadas + 10 + 6
log.info(f"Etapa 5 concluída! {df_cnpj_espacial.shape[1]} colunas (esperado: {esperado_final}): "
         f"{'confere' if df_cnpj_espacial.shape[1] == esperado_final else 'NÃO confere'}.")
log.info(f"Nomes de colunas sem repetição: {'confere' if df_cnpj_espacial.columns.is_unique else 'NÃO confere'}")

log.info(f"\nSEÇÃO 3 concluída! Tempo total: {formatar_tempo(time.time() - inicio_datas)}.")


## 4. Gravação e validação
Confere que a junção não mudou o número de linhas e grava a base final.


In [ ]:
log.info("SEÇÃO 4: gravação e validação...")
# A junção não pode mudar o número de linhas: um CNPJ por linha.
assert len(df_cnpj_espacial) == len(df_espacial), "a junção mudou o número de linhas"
validacao.validar_tabela("df_cnpj_espacial", df_cnpj_espacial, chave="cnpj", obrigatorias=["cnpj", "lat", "lon"],
                         linhas_entrada=len(df_espacial))
fontes.gravar_processamento(cfg, "df_cnpj_espacial", df_cnpj_espacial)
log.info("SEÇÃO 4 concluída!")

## 5. Resumo
Números principais, arquivos gerados e tempo total.


In [ ]:
log.info("RESUMO do 08_montar_base_final...")
log.info(f"df_cnpj_espacial: {len(df_cnpj_espacial):,} CNPJs | {df_cnpj_espacial.shape[1]} colunas")
log.info("Arquivos gerados:")
for chave in ['df_cnpj_espacial']:
    caminho = cfg.caminho("processamento." + chave)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")